Load data, prompt

In [15]:
import os
import json
import importlib
import pandas as pd
import numpy as np

from openai import OpenAI
from sklearn.metrics import accuracy_score, f1_score
import simpledorff

# ============================================================
# PROMPT
# ============================================================

import multiframe_prompts
importlib.reload(multiframe_prompts)

from multiframe_prompts import OPERATIONALIZED_FRAMES_PROMPT

PROMPT = OPERATIONALIZED_FRAMES_PROMPT

# ============================================================
# MODEL SETTINGS
# ============================================================

MODEL = "gpt-5"
REASONING_EFFORT = "medium"

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


# ============================================================
# LOAD GOLD DATA
# ============================================================

df = pd.read_excel("frame_gold.xlsx")

required_columns = [
    "uuid",
    "title",
    "protest_excerpt",
    "title_frame",
    "excerpt_frame",
]

missing = [c for c in required_columns if c not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_columns].copy()

df["title"] = df["title"].fillna("").astype(str)
df["protest_excerpt"] = df["protest_excerpt"].fillna("").astype(str)


# ============================================================
# NORMALIZE GOLD LABELS
# ============================================================

def normalize_frame(x):
    if pd.isna(x):
        return np.nan

    return (
        str(x)
        .strip()
        .lower()
        .replace(" ", "_")
    )


df["title_frame"] = df["title_frame"].apply(normalize_frame)
df["excerpt_frame"] = df["excerpt_frame"].apply(normalize_frame)

# In case source file still uses "violence"
df["title_frame"] = df["title_frame"].replace(
    "violence", "protester_violence"
)

df["excerpt_frame"] = df["excerpt_frame"].replace(
    "violence", "protester_violence"
)


# ============================================================
# TEST FIRST
# ============================================================

test_df = df.head(3).copy()

# Use this while testing:
RUN_DF = df

# Change to this for full validation:
# RUN_DF = df

print(f"Running {len(RUN_DF)} samples.")

display(RUN_DF.head(3))

Running 120 samples.


,uuid,title,protest_excerpt,title_frame,excerpt_frame
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,Third politician in a month killed in Ecuador,The gang boss was moved out of a jail with lig...,residual,grievance
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,Amazon CEO tells staff to work in office three...,This comes after Amazon said it would lay off ...,residual,widespread_support
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,Gaza ceasefire protest in Capitol rotunda lead...,Capitol rotunda on Sept. 27. Photo: Michael Ro...,grievance,disruption


Schema and API helper

In [17]:
# ============================================================
# LABELS
# ============================================================

FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
    "residual",
]

SECONDARY_FRAME_LABELS = [
    "protester_violence",
    "harm_to_protesters",
    "property_damage",
    "disruption",
    "grievance",
    "widespread_support",
    "peaceful",
    "impactful",
]


# ============================================================
# STRUCTURED OUTPUT SCHEMA
# ============================================================

FRAMES_SCHEMA = {
    "type": "object",
    "properties": {
        "title_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "excerpt_primary_frame": {
            "type": "string",
            "enum": FRAME_LABELS,
        },
        "excerpt_secondary_frame": {
            "anyOf": [
                {
                    "type": "string",
                    "enum": SECONDARY_FRAME_LABELS,
                },
                {
                    "type": "null"
                }
            ]
        },
    },
    "required": [
        "title_frame",
        "excerpt_primary_frame",
        "excerpt_secondary_frame",
    ],
    "additionalProperties": False,
}


# ============================================================
# API CALL
# ============================================================

def call_classifier(
    title,
    excerpt,
    system_prompt,
    schema,
    schema_name,
):
    
    user_input = f"""TITLE:
{title}

EXCERPT:
{excerpt}
"""

    try:

        response = client.responses.create(
            model=MODEL,
            reasoning={
                "effort": REASONING_EFFORT
            },
            instructions=system_prompt,
            input=user_input,
            text={
                "format": {
                    "type": "json_schema",
                    "name": schema_name,
                    "schema": schema,
                    "strict": True,
                }
            },
        )

        result = json.loads(response.output_text)

        # -----------------------------------------
        # Safety checks beyond JSON schema
        # -----------------------------------------

        primary = result["excerpt_primary_frame"]
        secondary = result["excerpt_secondary_frame"]

        # Primary Residual -> Secondary must be null
        if primary == "residual":
            secondary = None

        # Primary and Secondary cannot be identical
        if secondary == primary:
            secondary = None

        result["excerpt_secondary_frame"] = secondary

        # -----------------------------------------
        # Metadata
        # -----------------------------------------

        result["response_id"] = response.id

        usage = response.usage

        result["input_tokens"] = (
            usage.input_tokens
            if usage is not None
            else None
        )

        result["output_tokens"] = (
            usage.output_tokens
            if usage is not None
            else None
        )

        # Cached tokens
        result["cached_tokens"] = None

        if (
            usage is not None
            and getattr(usage, "input_tokens_details", None)
            is not None
        ):
            result["cached_tokens"] = getattr(
                usage.input_tokens_details,
                "cached_tokens",
                None,
            )

        # Reasoning tokens
        result["reasoning_tokens"] = None

        if (
            usage is not None
            and getattr(usage, "output_tokens_details", None)
            is not None
        ):
            result["reasoning_tokens"] = getattr(
                usage.output_tokens_details,
                "reasoning_tokens",
                None,
            )

        result["error"] = None

        return result

    except Exception as e:

        return {
            "title_frame": None,
            "excerpt_primary_frame": None,
            "excerpt_secondary_frame": None,
            "response_id": None,
            "input_tokens": None,
            "cached_tokens": None,
            "output_tokens": None,
            "reasoning_tokens": None,
            "error": str(e),
        }

RUN! and save

In [18]:
# ============================================================
# RUN FRAME CLASSIFICATION
# ============================================================

frame_results = []

print("\n==============================")
print("RUNNING FRAME CLASSIFICATION")
print("==============================")

for count, (_, row) in enumerate(RUN_DF.iterrows(), start=1):

    print(f"[{count}/{len(RUN_DF)}] {row['uuid']}")

    result = call_classifier(
        title=row["title"],
        excerpt=row["protest_excerpt"],
        system_prompt=PROMPT,
        schema=FRAMES_SCHEMA,
        schema_name="operationalized_multiframe",
    )

    result["uuid"] = row["uuid"]

    frame_results.append(result)


# ============================================================
# CREATE RESULTS DF
# ============================================================

frames_df = pd.DataFrame(frame_results)

frames_df = frames_df.rename(columns={
    "title_frame":
        "gpt_title_frame",

    "excerpt_primary_frame":
        "gpt_excerpt_primary_frame",

    "excerpt_secondary_frame":
        "gpt_excerpt_secondary_frame",

    "response_id":
        "frames_response_id",

    "input_tokens":
        "frames_input_tokens",

    "cached_tokens":
        "frames_cached_tokens",

    "output_tokens":
        "frames_output_tokens",

    "reasoning_tokens":
        "frames_reasoning_tokens",

    "error":
        "frames_error",
})


# ============================================================
# MERGE GOLD + GPT RESULTS
# ============================================================

results_df = RUN_DF.merge(
    frames_df,
    on="uuid",
    how="left"
)


# ============================================================
# CREATE CORRECTNESS VARIABLES
# ============================================================

# Title: only one prediction
results_df["title_correct"] = (
    results_df["title_frame"]
    ==
    results_df["gpt_title_frame"]
)


# Excerpt: strict Primary match
results_df["excerpt_primary_correct"] = (
    results_df["excerpt_frame"]
    ==
    results_df["gpt_excerpt_primary_frame"]
)


# Excerpt: Gold matches EITHER Primary OR Secondary
results_df["excerpt_primary_or_secondary_correct"] = (
    (
        results_df["excerpt_frame"]
        ==
        results_df["gpt_excerpt_primary_frame"]
    )
    |
    (
        results_df["excerpt_frame"]
        ==
        results_df["gpt_excerpt_secondary_frame"]
    )
)


# Optional diagnostic:
# Did secondary rescue an incorrect primary prediction?
results_df["secondary_rescue"] = (
    (~results_df["excerpt_primary_correct"])
    &
    (results_df["excerpt_primary_or_secondary_correct"])
)


# ============================================================
# SAVE
# ============================================================

results_df.to_excel(
    "operationalized_multiframe_results.xlsx",
    index=False
)

print("\nResults saved.")

display(
    results_df[
        [
            "uuid",
            "title_frame",
            "gpt_title_frame",
            "excerpt_frame",
            "gpt_excerpt_primary_frame",
            "gpt_excerpt_secondary_frame",
            "excerpt_primary_correct",
            "excerpt_primary_or_secondary_correct",
            "secondary_rescue",
        ]
    ]
)


RUNNING FRAME CLASSIFICATION
[1/120] d1334539-4cf3-4c5f-a0ef-88db5f167395
[2/120] b088e93f-1de4-4021-97bf-a90700b0a0ac
[3/120] fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad
[4/120] 155faae5-c8b6-441e-a20e-dce06e01cf32
[5/120] 50fea669-6828-44de-8a70-b448e398be11
[6/120] 81efb265-e3ba-4508-adc7-73cbc029f359
[7/120] 5fb219a8-9df7-4ad0-bc57-d46be3e327e0
[8/120] 252fb33d-5e6c-4cff-acc0-98c253f6c4b8
[9/120] 4acf5304-e74d-4a4f-bee0-caf65bd02abe
[10/120] 74fd09e2-35a3-495b-9b59-1610ec3398e1
[11/120] 7d9d161f-8087-4e0b-89e2-4a9bc1dcfd9d
[12/120] 0c5fa89f-f32f-4014-98e5-e8ad481b8a2f
[13/120] b2a2ef91-319a-47bf-9a61-1f8b4e192abf
[14/120] a92ef5b3-9891-4492-8a32-5024d7cc52dc
[15/120] c13350d6-ae7e-43f9-8999-17e530a4c80b
[16/120] 874c16ec-4684-40c7-8928-06baaa20260d
[17/120] 0192cd22-a165-4eb8-bf6d-da6165164b2e
[18/120] 95c2c809-f17e-4d9d-af74-26ba987929cd
[19/120] 109cd735-179b-4d53-ba32-ac1b70d5d990
[20/120] 36ab3319-2965-4075-80bd-14627327ab65
[21/120] f4bf7615-7d65-4e3d-b332-fbf590087f99
[22/120] 58a5

,uuid,title_frame,gpt_title_frame,excerpt_frame,gpt_excerpt_primary_frame,gpt_excerpt_secondary_frame,excerpt_primary_correct,excerpt_primary_or_secondary_correct,secondary_rescue
0,d1334539-4cf3-4c5f-a0ef-88db5f167395,residual,residual,grievance,grievance,NaN,True,True,False
1,b088e93f-1de4-4021-97bf-a90700b0a0ac,residual,residual,widespread_support,grievance,widespread_support,False,True,True
2,fffb4a3a-8ee2-4cc4-9b9a-b0f676c5bdad,grievance,residual,disruption,disruption,NaN,True,True,False
3,155faae5-c8b6-441e-a20e-dce06e01cf32,residual,residual,grievance,grievance,NaN,True,True,False
4,50fea669-6828-44de-8a70-b448e398be11,residual,residual,protester_violence,residual,NaN,False,False,False
...,...,...,...,...,...,...,...,...,...
115,559e0342-5214-4672-86dc-1c16db863422,harm_to_protesters,residual,harm_to_protesters,harm_to_protesters,impactful,True,True,False
116,94c2367b-5c21-4e9a-ae26-737c42c6a742,residual,residual,widespread_support,widespread_support,grievance,True,True,False
117,2ed7785a-1b12-4aa6-8210-6baf29d59351,residual,residual,disruption,protester_violence,disruption,False,True,True
118,c2b639e1-236d-4f8f-9c03-bda271d30d18,residual,residual,grievance,protester_violence,disruption,False,False,False


Summary table

In [19]:
# ============================================================
# KRIPPENDORFF ALPHA HELPER
# ============================================================

def calculate_alpha(gold, pred):

    temp = pd.DataFrame({
        "gold": gold,
        "gpt": pred,
    }).dropna()

    if len(temp) == 0:
        return np.nan

    temp["unit_id"] = range(len(temp))

    long_df = temp.melt(
        id_vars="unit_id",
        value_vars=["gold", "gpt"],
        var_name="annotator",
        value_name="label",
    )

    alpha = simpledorff.calculate_krippendorffs_alpha_for_df(
        long_df,
        experiment_col="unit_id",
        annotator_col="annotator",
        class_col="label",
    )

    return alpha


# ============================================================
# ONLY EVALUATE SUCCESSFUL RESPONSES
# ============================================================

eval_df = results_df[
    results_df["frames_error"].isna()
].copy()


# ============================================================
# TITLE METRICS
# ============================================================

title_accuracy = accuracy_score(
    eval_df["title_frame"],
    eval_df["gpt_title_frame"],
)

title_macro_f1 = f1_score(
    eval_df["title_frame"],
    eval_df["gpt_title_frame"],
    average="macro",
)

title_alpha = calculate_alpha(
    eval_df["title_frame"],
    eval_df["gpt_title_frame"],
)


# ============================================================
# EXCERPT: PRIMARY METRICS
# ============================================================

excerpt_primary_accuracy = accuracy_score(
    eval_df["excerpt_frame"],
    eval_df["gpt_excerpt_primary_frame"],
)

excerpt_macro_f1 = f1_score(
    eval_df["excerpt_frame"],
    eval_df["gpt_excerpt_primary_frame"],
    average="macro",
)

excerpt_alpha = calculate_alpha(
    eval_df["excerpt_frame"],
    eval_df["gpt_excerpt_primary_frame"],
)


# ============================================================
# EXCERPT: PRIMARY OR SECONDARY ACCURACY
# ============================================================

excerpt_primary_or_secondary_accuracy = (
    eval_df["excerpt_primary_or_secondary_correct"].mean()
)


# ============================================================
# SECONDARY-FRAME DIAGNOSTICS
# ============================================================

secondary_assignment_rate = (
    eval_df["gpt_excerpt_secondary_frame"]
    .notna()
    .mean()
)

secondary_rescue_count = (
    eval_df["secondary_rescue"].sum()
)

secondary_rescue_rate = (
    eval_df["secondary_rescue"].mean()
)


# ============================================================
# SUMMARY TABLE
# ============================================================

summary_df = pd.DataFrame({
    "Task": [
        "Title Frame",
        "Excerpt Frame",
    ],

    "N": [
        len(eval_df),
        len(eval_df),
    ],

    "Primary Accuracy": [
        title_accuracy,
        excerpt_primary_accuracy,
    ],

    "Primary or Secondary Accuracy": [
        np.nan,
        excerpt_primary_or_secondary_accuracy,
    ],

    "Macro F1": [
        title_macro_f1,
        excerpt_macro_f1,
    ],

    "Krippendorff Alpha": [
        title_alpha,
        excerpt_alpha,
    ],
})


metric_cols = [
    "Primary Accuracy",
    "Primary or Secondary Accuracy",
    "Macro F1",
    "Krippendorff Alpha",
]

summary_df[metric_cols] = (
    summary_df[metric_cols]
    .round(3)
)

display(summary_df)


# ============================================================
# SECONDARY-FRAME SUMMARY
# ============================================================

print("\nSecondary-frame diagnostics:")
print(
    f"Secondary assigned: "
    f"{secondary_assignment_rate:.3f}"
)

print(
    f"Errors rescued by secondary: "
    f"{secondary_rescue_count}"
)

print(
    f"Rescue rate across all excerpts: "
    f"{secondary_rescue_rate:.3f}"
)

,Task,N,Primary Accuracy,Primary or Secondary Accuracy,Macro F1,Krippendorff Alpha
0,Title Frame,120,0.783,NaN,0.618,0.603
1,Excerpt Frame,120,0.650,0.808,0.627,0.587



Secondary-frame diagnostics:
Secondary assigned: 0.542
Errors rescued by secondary: 19
Rescue rate across all excerpts: 0.158


In [20]:
summary_df.to_excel(
    "operationalized_multiframe_summary.xlsx",
    index=False
)